In [51]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


In [52]:
## loading model from dotenv

import os
from dotenv import load_dotenv,dotenv_values
load_dotenv()

m2 = os.getenv("MODEL_3")
print(m2)

llama3.1:8b


In [53]:
#model = OllamaLLM(model="llama3.1:8b")    ##directly use model name without dotenv
model = OllamaLLM(model=m2)

In [54]:
template = """
You are an expert in recommendation for the restaurant named "Pizza Place" and thus will be asked about various type of restaurants and where the best type of particular food will be available, etc.
You might also be expected to plan some retaurants based on rough iternary and thu, you would be expected to recommend multiple restaurants at a time. 

Example:
User: " We are going to a bar and then would be going to a have dinner after it, 
can you recommend some bar that bar that ha good beer and BBQ restaurant?"
Expected Workflow: First search for the firt type of restaurant and then look for the other and recommend them."

If the user asks for a particular style of pizza/dish, search for the pizza or dish that fits the description and include its name in the result. 
If you cannot find the appropriate result, inform the user about it.

Example: 
User: Do they have any vegan pizza.
Agent-action: 1.) first search for the required pizza, in this case you got a result with name "tropical blast".
If you also obtain the name of the dish/ pizza, be sure to include the name of the pizza/dish in the final result.


You should not be overly formal but at the ame time, keep your answers simple and easy to understand as the users typically prefer short answer with a touch of familiarity,
as if they are having a conversation with friend.
Do not hallucinate and give at-most 3 recomendation with the information such a the place's name .

Here are some relevant reviews: {reviews}

Here is the question to answer: {questions}
"""
prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

##test
chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":"What is the best pizza place in town?"})

"You're looking for the best pizza place in town, huh? Based on the reviews, I'd say Lafamilia is the top pick! They're known for serving amazing pizzas, and it seems like they're a local favorite. If you're craving a great pizza, I'd recommend checking them out!"

In [55]:
## Write in a python script
'''
while True:
    question = input("Ask your question (q to quit):")
    if question == "q":
        break
    
    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})
    
'''

'\nwhile True:\n    question = input("Ask your question (q to quit):")\n    if question == "q":\n        break\n\n    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})\n\n'

In [56]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
import os
import pandas as pd

In [57]:
dir = "data/csv/RRr.csv"
df = pd.read_csv(dir)

In [58]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:v1")

In [59]:
db_location = "./chrome_langchain_db"
add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    ids = []
    
    for i, row in df.iterrows():
        document = Document(
            page_content = row["Title"] + " " + row["Review"],
            metadata = {"rating": row["Rating"], "date":row["Date"]},
            id= str(i)
        )
        ids.append(str(i))
        documents.append(document)
    
    
vector_store = Chroma(
    collection_name = "restaurant_reviews",
    persist_directory=db_location,
    embedding_function = embeddings
)
        
if add_documents:
    vector_store.add_documents(documents=documents,ids = ids)


In [60]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k":5}
)

In [61]:
import re
regex_pattern = r"<think>[\s\S]*?<\/think>\n?\n?"


In [62]:
question = "Do they have a roman style pizza?"

reviews = retriever.invoke(question)

result = chain.invoke({"reviews":reviews,"questions":question})
#cleaned_response = re.sub(regex_pattern, '', result)
print(result) # Output: This is the actual answer


Based on the reviews, I found a mention of Roman-style pizza in review #1. The reviewer mentioned that the Roman-style pizza al taglio (rectangular, with a focaccia-like base) is a welcome alternative to the usual round pies.

So, yes! They do have a Roman-style pizza. 

Recommendation:
1. Try their Roman-style pizza al taglio.
2. You can also consider their other options, like the wood-fired Margherita pizza (review #2).
3. If you want something a bit different, go for their fig and prosciutto pizza, which has been mentioned as excellent in review #4.
